# UCI Student Performance — EDA & ML
**Dataset:** Paulo Cortez, 2008. 395 Math + 649 Portuguese = 1,044 students.
**Use case:** Early Academic Warning System — predict dropout risk after midterm G1/G2.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

In [ ]:
# ── Cell 1: Load & combine ──────────────────────────────────────
mat = pd.read_csv('../data/raw/student-mat.csv', sep=';')
por = pd.read_csv('../data/raw/student-por.csv', sep=';')
mat['subject'] = 'math'
por['subject'] = 'portuguese'
df = pd.concat([mat, por], ignore_index=True)
print(f'Combined shape: {df.shape}')  # (1044, 34)
df.head(3)

In [ ]:
# ── Cell 2: Clean & feature engineering ────────────────────────
df.columns = df.columns.str.lower()
df = df.rename(columns={
    'medu':'mother_edu', 'fedu':'father_edu',
    'mjob':'mother_job', 'fjob':'father_job',
    'dalc':'alcohol_weekday', 'walc':'alcohol_weekend',
    'g1':'grade_mid1', 'g2':'grade_mid2', 'g3':'grade_final'
})

# Binary target: at_risk if grade_final < 10 (Portuguese pass threshold)
df['at_risk'] = (df['grade_final'] < 10).astype(int)

# Encode binary yes/no features for ML
for col in ['schoolsup','famsup','paid','activities','internet','romantic','nursery','higher']:
    df[col] = df[col].str.lower().map({'yes':1,'no':0})

# Keep categorical strings for frontend filtering
df['sex']    = df['sex'].str.lower()     # 'f' / 'm'
df['school'] = df['school'].str.lower()  # 'gp' / 'ms'
df['address']= df['address'].str.lower() # 'u' / 'r'
df['pstatus']= df['pstatus'].str.lower() # 't' / 'a'
df['famsize']= df['famsize'].str.lower() # 'le3' / 'gt3'

print(f'At-risk rate: {df["at_risk"].mean():.1%} ({df["at_risk"].sum()} students)')
print(f'G3 = 0 (withdrew): {(df["grade_final"]==0).sum()} students')
df[['grade_mid1','grade_mid2','grade_final','at_risk']].describe().round(2)

In [ ]:
# ── Cell 3: G3 distribution (bimodal) ──────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col, title in zip(axes,
    ['grade_mid1','grade_mid2','grade_final'],
    ['G1 (Period 1)','G2 (Period 2)','G3 (Final)']):
    ax.hist(df[col], bins=20, color='steelblue', alpha=0.7, edgecolor='white')
    ax.axvline(10, color='red', linestyle='--', alpha=0.7, label='Pass threshold')
    ax.set_title(f'{title}\nMean={df[col].mean():.1f}, Std={df[col].std():.1f}')
    ax.set_xlabel('Grade (0–20)')
plt.suptitle('Grade Distribution Across 3 Periods', y=1.02, fontsize=13)
plt.tight_layout()
plt.savefig('../visual/grade_distributions.png', bbox_inches='tight')
plt.show()
print('NOTE: G3=0 spike = students who withdrew / did not sit final exam')

In [ ]:
# ── Cell 4: Pearson correlations vs grade_final ─────────────────
numeric_feats = ['grade_mid1','grade_mid2','absences','failures','studytime',
                 'mother_edu','father_edu','goout','alcohol_weekday','alcohol_weekend',
                 'health','freetime','famrel','age','traveltime']
corrs = df[numeric_feats].corrwith(df['grade_final']).sort_values(key=abs, ascending=False)
print('=== Pearson r vs grade_final ===')
print(corrs.round(4))

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#16a34a' if v >= 0 else '#dc2626' for v in corrs]
corrs.plot.barh(ax=ax, color=colors, alpha=0.8)
ax.axvline(0, color='gray', linewidth=0.8)
ax.set_xlabel('Pearson r (correlation with G3)')
ax.set_title('Feature Correlations with Final Grade')
plt.tight_layout()
plt.savefig('../visual/correlations.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Cell 5: Grade progression regression ────────────────────────
for prev, curr in [('grade_mid1','grade_mid2'), ('grade_mid2','grade_final')]:
    X = sm.add_constant(df[prev])
    model = sm.OLS(df[curr], X).fit()
    r = df[prev].corr(df[curr])
    print(f'{prev} → {curr}:')
    print(f'  r={r:.3f}  R²={model.rsquared:.3f}  β={model.params[prev]:.4f}  p={model.pvalues[prev]:.2e}')

# Scatter G2 vs G3 (strongest relationship)
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(df['grade_mid2'], df['grade_final'], alpha=0.15, s=20, color='steelblue')
m, b = np.polyfit(df['grade_mid2'], df['grade_final'], 1)
xs = np.linspace(0, 20, 100)
ax.plot(xs, m*xs+b, 'r--', linewidth=2, label=f'y={m:.2f}x+{b:.2f}  r={df["grade_mid2"].corr(df["grade_final"]):.3f}')
ax.axvline(10, color='orange', linestyle=':', alpha=0.6)
ax.axhline(10, color='orange', linestyle=':', alpha=0.6)
ax.set_xlabel('Midterm 2 (G2)'); ax.set_ylabel('Final Grade (G3)')
ax.set_title('G2 vs G3: r ≈ 0.91 — strongest predictor')
ax.legend()
plt.savefig('../visual/g2_vs_g3.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Cell 6: K-Means clustering (k=3) ────────────────────────────
cluster_cols = ['studytime', 'absences', 'goout', 'alcohol_weekend']
X_c = df[cluster_cols].copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_c)

# Elbow method
inertias = []
for k in range(2, 8):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
plt.plot(range(2,8), inertias, 'o-', color='steelblue')
plt.xlabel('K'); plt.ylabel('Inertia'); plt.title('Elbow Method — K=3 chosen')
plt.savefig('../visual/elbow.png', bbox_inches='tight')
plt.show()

# Final K=3
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X_scaled)

result = df.groupby('cluster').agg({
    'studytime':       'mean',
    'absences':        'mean',
    'goout':           'mean',
    'alcohol_weekend': 'mean',
    'grade_final':     ['mean','count'],
    'at_risk':         'mean'
}).round(2)
print(result)
print('\nCluster interpretation:')
print('  Cluster 1 (high study): Focused Achievers — best grades, lowest risk')
print('  Cluster 0 (balanced):   Average Learners — moderate performance')
print('  Cluster 2 (high social): Social Risk Group — highest absences/alcohol, worst risk')

In [ ]:
# ── Cell 7: Random Forest — Early Warning Classifier ─────────────
# Features available at time of prediction (after G2 midterm)
# NOTE: grade_final is TARGET — not used as feature (no leakage)
feature_cols = [
    'grade_mid1', 'grade_mid2',           # midterm results
    'absences', 'failures', 'studytime',  # behavior
    'mother_edu', 'father_edu',           # socioeconomic
    'goout', 'alcohol_weekend', 'romantic',
    'internet', 'schoolsup', 'activities'
]
X = df[feature_cols]
y = df['at_risk']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

clf = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',  # minority class (at_risk) gets more weight
    random_state=42, n_jobs=-1
)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print(classification_report(y_test, y_pred, target_names=['Pass','At Risk']))
cm = confusion_matrix(y_test, y_pred)
print('Confusion Matrix:')
print(pd.DataFrame(cm, index=['Actual: Pass','Actual: Risk'], columns=['Pred: Pass','Pred: Risk']))

# Cross-validation recall
cv = cross_val_score(clf, X, y, cv=5, scoring='recall')
print(f'\nCV Recall (5-fold): {cv.mean():.3f} ± {cv.std():.3f}')

# Feature importance
fi = pd.Series(clf.feature_importances_, index=feature_cols).sort_values(ascending=False)
print('\nTop feature importances:')
print(fi.head(8).round(4))

In [ ]:
# ── Cell 8: Export clean CSV ─────────────────────────────────────
export_cols = [
    'school','sex','age','address','famsize','pstatus',
    'mother_edu','father_edu','mother_job','father_job','reason','guardian',
    'traveltime','studytime','failures','schoolsup','famsup','paid',
    'activities','nursery','higher','internet','romantic',
    'famrel','freetime','goout','alcohol_weekday','alcohol_weekend',
    'health','absences','grade_mid1','grade_mid2','grade_final',
    'at_risk','subject','cluster'
]
df[export_cols].to_csv('../data/processed/clean_students.csv', index=False)

import shutil
shutil.copy('../data/processed/clean_students.csv', '../web/data/clean_students.csv')

print(f'Exported {len(df)} rows × {len(export_cols)} cols')
print('→ data/processed/clean_students.csv')
print('→ web/data/clean_students.csv')
print('\nNOW: Run the dashboard with:')
print('  python -m http.server 8000 --directory web')
print('  Open: http://localhost:8000')